# Fase B — Limpieza y normalización de datos

**Proyecto 2 y 3 · Siniestralidad en Cali 2016–2024**

- **Entrada:** `siniestralidad_2016_2024.csv` (archivo fuente).
- **Salida:** `entrega/datos_limpios.parquet`, listo para la Fase C (modelado).

Este notebook cubre el pipeline completo: diagnóstico, limpieza estructural, normalización de texto, tratamiento de nulos y atípicos, codificación (Ordinal y One-Hot), transformación logarítmica y criterio de escalado. Cada decisión lleva su justificación.

> **Regla de no fuga de información.** El escalado no se ajusta sobre todo el dataset en esta fase. Un escalador ajustado con todos los datos conoce estadísticos del conjunto de prueba (*data leakage*). Por eso aquí se elige y justifica el escalador, y se ajusta dentro de un `Pipeline` en la Fase C, después del `train_test_split`.

## 1. Carga y auditoría inicial

In [ ]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", 50)

RUTAS_CANDIDATAS = [
    Path("../../siniestralidad_2016_2024.csv"),
    Path("../../../siniestralidad_2016_2024.csv"),
]
RUTA_CSV = next((ruta for ruta in RUTAS_CANDIDATAS if ruta.exists()), None)

if RUTA_CSV is None:
    raise FileNotFoundError(
        "No se encontró siniestralidad_2016_2024.csv. Rutas probadas: "
        + ", ".join(str(r.resolve()) for r in RUTAS_CANDIDATAS)
    )


RUTA_SALIDA = Path("datos_limpios.parquet")
COLUMNA_OBJETIVO = "con_victimas"
UMBRAL_ACUERDO = 0.90  # acuerdo mínimo para recuperar numero_vehiculos por regla de negocio

df_raw = pd.read_csv(
    RUTA_CSV,
    sep=";",
    encoding="latin-1",
    dtype="string",
    na_values=[".", ""],
    low_memory=False,
)

print(f"Archivo: {RUTA_CSV.resolve()}")
print(f"Registros: {len(df_raw):,} | Columnas: {df_raw.shape[1]}")
display(df_raw.head())

Archivo: /home/santiago/Documentos/Universidad/Ciencia de Datos/siniestralidad_2016_2024.csv
Registros: 371,003 | Columnas: 10


,Código,Fecha,Tipo confirmado,Dirección reporte,Tipo clase de accidente,Tipo clase de accidente.1,Tipo confirmado.1,Medio de reporte,Numero vehículos,Tipo de vehículos implicados
0,96159,1/01/2016,Solo daños,Carrera 56 Con Calle 1 A,<NA>,<NA>,Solo Da?os,RADIO,2,MOTOCICLETA
1,96160,1/01/2016,Negativo,Calle 72 O Con Carrera 28 H,<NA>,<NA>,Negativo,RADIO,<NA>,<NA>
2,96161,1/01/2016,Con lesionado,Carrera 94 Con Calle 4 C,ATROPELLO,ATROPELLO,Por Atropello,TELEF?NICO 127,2,MOTOCICLETA
3,96162,1/01/2016,Con lesionado,Carrera 8 A Con Calle 44,VOLCAMIENTO,VOLCAMIENTO,Por Volcamiento,RADIO,1,MOTOCICLETA
4,96163,1/01/2016,Negativo,Carrera 28 H Con Calle 72 M,<NA>,<NA>,Negativo,TELEF?NICO 127,<NA>,<NA>


## 2. Diagnóstico

Hallazgos de la auditoría previa sobre el archivo fuente (371.003 registros, 10 columnas):

- **Duplicados exactos:** 4 filas, correspondientes a 2 pares de códigos repetidos.
- **Registro vacío:** 1 fila sin código, fecha ni valores.
- **Columnas duplicadas:** `Tipo clase de accidente` y `Tipo clase de accidente.1` coinciden al 100 %, así que son redundantes. En cambio, `Tipo confirmado` y `Tipo confirmado.1` **no** son redundantes: la segunda usa otro catálogo y mezcla gravedad con tipo de accidente (`Por Atropello`, `Inspeccion a Cadaver`).
- **Nulos altos:** `Tipo clase de accidente` 87,7 %, `Medio de reporte` 77,7 %, `Tipo de vehículos` 59,9 %, `Numero vehículos` 52,4 % y `Dirección reporte` 49,5 %.
- **Inconsistencias de texto:** una misma intersección aparece como `CARRERA 56 CON CALLE 1 A` y como `CALLE 36 & CARRERA 47`; el medio de reporte se escribe de varias formas (`RADIO`, `AGENTE RADIO`); y el archivo trae `?` donde van letras acentuadas (`TELEF?NICO`, `DA?OS`).
- **Error corregido:** el notebook anterior convertía la fecha sin formato explícito y dejaba ~224.700 fechas en NaT (cerca del 60 %). Aquí el formato se fija como `%d/%m/%Y`.

In [14]:
diagnostico = pd.DataFrame({
    "tipo": df_raw.dtypes.astype(str),
    "nulos": df_raw.isna().sum(),
    "% nulos": (df_raw.isna().mean() * 100).round(2),
    "valores_unicos": df_raw.nunique(dropna=True),
})
display(diagnostico)

print("Filas completamente vacías:", int(df_raw.isna().all(axis=1).sum()))
print("Duplicados exactos:", int(df_raw.duplicated().sum()))
print("Códigos nulos:", int(df_raw["Código"].isna().sum()))
print("Códigos repetidos:", int(df_raw["Código"].dropna().duplicated().sum()))

# ¿Las columnas duplicadas son redundantes o son catálogos distintos?
for col_a, col_b in [
    ("Tipo clase de accidente", "Tipo clase de accidente.1"),
    ("Tipo confirmado", "Tipo confirmado.1"),
]:
    ambas = df_raw[[col_a, col_b]].dropna()
    coincidencia = (ambas[col_a] == ambas[col_b]).mean()
    print(f"{col_a} vs {col_b}: coincidencia {coincidencia:.2%} sobre {len(ambas):,} filas con ambas")

print("\nValores de 'Tipo confirmado.1' (catálogo distinto a 'Tipo confirmado'):")
display(df_raw["Tipo confirmado.1"].value_counts(dropna=False))

# ¿La ausencia de medio de reporte depende del año? (define si es un nulo estructural)
nulos_medio_por_anio = (
    df_raw["Medio de reporte"].isna()
    .groupby(df_raw["Fecha"].str[-4:])
    .mean()
    .mul(100)
    .round(1)
)
print("\n% de nulos en 'Medio de reporte' por año:")
display(nulos_medio_por_anio)

,tipo,nulos,% nulos,valores_unicos
Código,string,1,0.00,370998
Fecha,string,1,0.00,3229
Tipo confirmado,string,3,0.00,6
Dirección reporte,string,183686,49.51,65334
Tipo clase de accidente,string,325258,87.67,14
Tipo clase de accidente.1,string,325258,87.67,14
Tipo confirmado.1,string,23389,6.30,21
Medio de reporte,string,288099,77.65,17
Numero vehículos,string,194424,52.40,12
Tipo de vehículos implicados,string,222214,59.90,843


Filas completamente vacías: 1
Duplicados exactos: 4
Códigos nulos: 1
Códigos repetidos: 4
Tipo clase de accidente vs Tipo clase de accidente.1: coincidencia 100.00% sobre 45,745 filas con ambas
Tipo confirmado vs Tipo confirmado.1: coincidencia 61.10% sobre 347,612 filas con ambas

Valores de 'Tipo confirmado.1' (catálogo distinto a 'Tipo confirmado'):


Tipo confirmado.1
Negativo                                         212387
Solo Da?os                                        34425
Arreglo Formal                                    28057
Choque con Lesionados                             26793
<NA>                                              23389
No Quiere Intervenci?n                            20981
NEGATIVO                                           7424
Por Atropello                                      4324
Falsa Alarma                                       3775
CON HERIDOS                                        2490
POSITIVO                                           2353
Inspeccion a Cadaver                               1656
Por Volcamiento                                    1093
Por Caida de Ocupante                               906
Muerto en la via                                    439
SOLO DAÑOS                                          185
CON MUERTOS                                         179
NoQuiereIntervenci?n          


% de nulos en 'Medio de reporte' por año:


Fecha
2016     99.5
2017    100.0
2018    100.0
2019    100.0
2020    100.0
2021     32.8
2022      0.0
2023      0.0
2024      1.8
Name: Medio de reporte, dtype: float64

## 3. Limpieza estructural

Orden de operaciones: primero se eliminan filas inválidas y duplicadas; después se fusionan las columnas duplicadas. Para la gravedad se usa `Tipo confirmado` y solo si está vacía `Tipo confirmado.1`. La sección 5 descarta los valores de esa segunda columna que no describen gravedad.

In [15]:
df = df_raw.copy()

# 1) Filas completamente vacías
mascara_vacias = df.isna().all(axis=1)
print("Filas vacías eliminadas:", int(mascara_vacias.sum()))
df = df.loc[~mascara_vacias]

# 2) Registros sin código de siniestro
print("Registros sin código eliminados:", int(df["Código"].isna().sum()))
df = df.loc[df["Código"].notna()]

# 3) Duplicados exactos
n_antes = len(df)
df = df.drop_duplicates()
print("Duplicados exactos eliminados:", n_antes - len(df))

# 4) Un solo registro por código de siniestro: se conserva el más completo
df = (
    df.assign(_completitud=df.notna().sum(axis=1))
    .sort_values("_completitud", ascending=False, kind="stable")
    .drop_duplicates(subset="Código", keep="first")
    .drop(columns="_completitud")
    .reset_index(drop=True)
)
print("Códigos repetidos tras la limpieza:", int(df["Código"].duplicated().sum()))
print(f"Registros tras la limpieza estructural: {len(df):,}")

Filas vacías eliminadas: 1
Registros sin código eliminados: 0
Duplicados exactos eliminados: 4
Códigos repetidos tras la limpieza: 0
Registros tras la limpieza estructural: 370,998


## 4. Normalización de texto y estandarización de categorías

Reglas uniformes para todo el texto: espacios sobrantes, mayúsculas, eliminación de tildes y notación de intersección (`&` pasa a `CON`). El `?` del archivo fuente se conserva, y las reglas de categorías usan `.` para aceptarlo (`TELEF.NICO` coincide con `TELEF?NICO`).

In [16]:
def quitar_tildes(valor):
    descompuesto = unicodedata.normalize("NFKD", valor)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))


def normalizar_texto(serie):
    # Quita espacios sobrantes, colapsa espacios internos, pasa a mayúsculas y elimina tildes.
    limpio = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )
    catalogo = {valor: quitar_tildes(valor) for valor in limpio.dropna().unique()}
    return limpio.map(catalogo).astype("string")


def estandarizar(serie, condiciones, etiquetas, defecto):
    # Las condiciones se evalúan en orden: la primera que se cumple gana.
    return pd.Series(np.select(condiciones, etiquetas, default=defecto), index=serie.index)


# --- Textos normalizados ---
df["direccion"] = normalizar_texto(df["Dirección reporte"])
df["direccion"] = df["direccion"].str.replace(r"\s*&\s*", " CON ", regex=True)
df["medio_texto"] = normalizar_texto(df["Medio de reporte"])
df["accidente_texto"] = normalizar_texto(
    df["Tipo clase de accidente"].fillna(df["Tipo clase de accidente.1"])
)
df["vehiculos_texto"] = normalizar_texto(df["Tipo de vehículos implicados"])

print("Valores distintos antes → después de normalizar:")
print(f"  Dirección:         {df['Dirección reporte'].nunique():>7,} → {df['direccion'].nunique():>7,}")
print(f"  Medio de reporte:  {df['Medio de reporte'].nunique():>7,} → {df['medio_texto'].nunique():>7,}")
print(f"  Tipo de vehículos: {df['Tipo de vehículos implicados'].nunique():>7,} → {df['vehiculos_texto'].nunique():>7,}")

# --- Tipo de accidente (nominal) ---
acc = df["accidente_texto"]
df["tipo_accidente"] = estandarizar(
    acc,
    [
        acc.isna(),
        acc.str.contains("ATROPEL", na=False),
        acc.str.contains("VOLCAM", na=False),
        acc.str.contains("CHOQUE", na=False),
        acc.str.contains("CA.DA.*OCUPANTE", na=False),
        acc.str.contains("INCEND", na=False),
    ],
    ["SIN_DATO", "ATROPELLO", "VOLCAMIENTO", "CHOQUE", "CAIDA_OCUPANTE", "INCENDIO"],
    defecto="OTRO",
)

# --- Medio de reporte (nominal) ---
# "AGENTE RADIO" se clasifica como RADIO: es el canal por el que se reportó.
medio = df["medio_texto"]
df["medio_reporte"] = estandarizar(
    medio,
    [
        medio.isna(),
        medio.str.contains("123", na=False),
        medio.str.contains("127", na=False),
        medio.str.contains("RADIO", na=False),
        medio.str.contains("AGENTE", na=False),
        medio.str.contains("REDES", na=False),
    ],
    ["SIN_DATO", "LINEA_123", "LINEA_127", "RADIO", "AGENTE_TRANSITO", "REDES_SOCIALES"],
    defecto="OTRO",
)
print("\nValores de medio que cayeron en OTRO (revisar):")
display(df.loc[df["medio_reporte"].eq("OTRO"), "medio_texto"].value_counts())

# --- Tipología de la dirección (nominal) ---
direccion = df["direccion"]
df["tipo_via_principal"] = estandarizar(
    direccion,
    [
        direccion.isna(),
        direccion.str.startswith("CARRERA", na=False),
        direccion.str.startswith("CALLE", na=False),
        direccion.str.startswith("DIAGONAL", na=False),
        direccion.str.startswith("TRANSVERSAL", na=False),
        direccion.str.startswith("AVENIDA", na=False),
    ],
    ["SIN_DATO", "CARRERA", "CALLE", "DIAGONAL", "TRANSVERSAL", "AVENIDA"],
    defecto="REFERENCIA",  # lugares con nombre propio: clínicas, CAI, estaciones
)
df["es_interseccion"] = direccion.str.contains(r"\bCON\b", regex=True, na=False).astype("int8")

print("\nDirecciones clasificadas como REFERENCIA (revisar):")
display(direccion[df["tipo_via_principal"].eq("REFERENCIA")].value_counts().head(10))

Valores distintos antes → después de normalizar:
  Dirección:          65,334 →  57,138
  Medio de reporte:       17 →      17
  Tipo de vehículos:     843 →     843

Valores de medio que cayeron en OTRO (revisar):


medio_texto
MOVILIDAD CALI                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       22
OTRO VALOR                                                                                                                                                                                                                                                                                                                                          


Direcciones clasificadas como REFERENCIA (revisar):


direccion
CLINICA COLOMBIA                   604
CLINICA CRISTO REY                 402
CLINICA VALLE DEL LILI             167
CLINICA REY DAVID                  140
CLINICA IMBANACO                   117
SELECCIONE... CON SELECCIONE...    113
CL?NICA REY DAVID                  100
CLINICA SANTA CLARA                 84
CLINICA SAN FERNANDO                81
VIA AL MAR                          67
Name: count, dtype: int64[pyarrow]

## 5. Fechas y variable objetivo

**Fecha:** se interpreta con formato explícito `%d/%m/%Y`. Los valores que no se pueden leer quedan como NaT y se eliminan.

**Variable objetivo `con_victimas`:**
- `1`: la gravedad registra lesionados, heridos o fallecidos.
- `0`: negativo, solo daños o arreglo formal.
- Sin valor: cualquier otro texto (por ejemplo, `Por Atropello`, que describe el tipo de accidente y no la gravedad). Estos registros se eliminan.

`tipo_confirmado` no se usa como predictor: es la fuente del objetivo, y dejarla en el modelo sería fuga de información.

In [17]:
df["fecha"] = pd.to_datetime(df["Fecha"], format="%d/%m/%Y", errors="coerce")

# Gravedad: columna principal; si está vacía, se usa la secundaria
gravedad = normalizar_texto(df["Tipo confirmado"].fillna(df["Tipo confirmado.1"]))
df[COLUMNA_OBJETIVO] = np.select(
    [
        gravedad.str.contains("LESION|HERID|MUERT|FALLEC|CADAVER", na=False),
        gravedad.str.contains("NEGATIV|SOLO DA|ARREGLO", na=False),
    ],
    [1, 0],
    default=-1,
)

print("Fechas no interpretables:", int(df["fecha"].isna().sum()))
print("Gravedad no interpretable:", int((df[COLUMNA_OBJETIVO] == -1).sum()))

antes = len(df)
df = df.loc[(df[COLUMNA_OBJETIVO] != -1) & df["fecha"].notna()].reset_index(drop=True)
df[COLUMNA_OBJETIVO] = df[COLUMNA_OBJETIVO].astype("int8")
print(f"Registros eliminados por objetivo o fecha inválidos: {antes - len(df):,}")

print("\nDistribución de la variable objetivo:")
display(df[COLUMNA_OBJETIVO].value_counts(normalize=True).round(4))

Fechas no interpretables: 0
Gravedad no interpretable: 0
Registros eliminados por objetivo o fecha inválidos: 0

Distribución de la variable objetivo:


con_victimas
0    0.8629
1    0.1371
Name: proportion, dtype: float64

## 6. Tratamiento de nulos

Estrategias según el tipo de variable:

| Variable | Tipo | Estrategia | Justificación |
|---|---|---|---|
| `tipo_accidente`, `medio_reporte`, `tipo_via_principal`, `tipos_vehiculos` | Nominal | Categoría explícita `SIN_DATO` | La ausencia es informativa: el dato no se registró. La moda sería justamente "sin dato" y no aportaría información. |
| `numero_vehiculos` | Numérica discreta y sesgada | (1) Recuperar por regla de negocio: el número de tipos listados en `Tipo de vehículos implicados`, si el acuerdo con el dato real es ≥ 90 %. (2) El resto, con la mediana. | Casi siempre el número de tipos listados coincide con el número de vehículos. La mediana es robusta ante el sesgo; la media daría valores no enteros. |
| `fecha` | Temporal | Eliminar el registro | Imputar una fecha inventaría un suceso. Es un caso aislado. |
| `con_victimas` | Objetivo | Eliminar el registro | Imputar la variable que se quiere predecir introduce ruido en el entrenamiento. |
| `dirección` | Nominal de alta cardinalidad | `SIN_DATO` y luego se extrae la tipología | Con cerca de 59 mil valores distintos, el texto crudo no sirve como categoría. |

KNN se consideró y se descartó: con ~371 mil filas y ~60 % de nulos en algunas columnas es costoso, y las variables no son continuas, así que las distancias carecen de sentido.

In [18]:
# (1) numero_vehiculos: recuperación por regla de negocio
df["numero_vehiculos"] = pd.to_numeric(df["Numero vehículos"], errors="coerce").astype("float64")
n_tipos_listados = (df["Tipo de vehículos implicados"].str.count(",") + 1).astype("float64")

pares = pd.DataFrame({"numero": df["numero_vehiculos"], "tipos": n_tipos_listados}).dropna()
tasa_acuerdo = (pares["numero"] == pares["tipos"]).mean()
print(f"Acuerdo entre numero_vehiculos y tipos listados: {tasa_acuerdo:.2%} (sobre {len(pares):,} filas)")

if tasa_acuerdo >= UMBRAL_ACUERDO:
    recuperables = df["numero_vehiculos"].isna() & n_tipos_listados.notna()
    df.loc[recuperables, "numero_vehiculos"] = n_tipos_listados[recuperables]
    print(f"Recuperados por regla de negocio: {int(recuperables.sum()):,}")
else:
    print("Acuerdo insuficiente: no se aplica la recuperación.")

# (2) El resto: mediana (variable discreta y sesgada)
pendientes = int(df["numero_vehiculos"].isna().sum())
mediana_vehiculos = df["numero_vehiculos"].median()
df["numero_vehiculos"] = df["numero_vehiculos"].fillna(mediana_vehiculos)
print(f"Imputados con la mediana ({mediana_vehiculos}): {pendientes:,}")
print("Nulos en numero_vehiculos:", int(df["numero_vehiculos"].isna().sum()))

Acuerdo entre numero_vehiculos y tipos listados: 90.15% (sobre 148,674 filas)
Recuperados por regla de negocio: 114
Imputados con la mediana (2.0): 194,307
Nulos en numero_vehiculos: 0


## 7. Detección y tratamiento de atípicos

Se aplican dos métodos:

- **IQR:** para `numero_vehiculos` el IQR es 0 (Q1 = Q3 = 2), porque la mayoría de los siniestros tiene dos vehículos. El método marcaría como atípico cualquier valor distinto de 2 y no sirve como criterio.
- **Z-score sobre log1p:** detecta los extremos reales. Se marcan los registros con |z| > 3.

**Decisión: conservar.** Los valores altos (3 a 16 vehículos) corresponden a choques múltiples reales, no a errores de captura. Eliminarlos quitaría justamente los siniestros más graves, que son los que más se relacionan con víctimas, y el modelo quedaría sesgado.

In [19]:
def resumen_iqr(serie, nombre):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inferior, limite_superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    atipicos = (serie < limite_inferior) | (serie > limite_superior)
    print(f"[{nombre}] Q1={q1}, Q3={q3}, IQR={iqr}")
    print(f"[{nombre}] Rango: [{limite_inferior}, {limite_superior}] → atípicos: {int(atipicos.sum()):,} ({atipicos.mean():.2%})")
    return atipicos


resumen_iqr(df["numero_vehiculos"], "numero_vehiculos")

log_vehiculos = np.log1p(df["numero_vehiculos"])
z_vehiculos = (log_vehiculos - log_vehiculos.mean()) / log_vehiculos.std()
atipicos_z = z_vehiculos.abs() > 3
print(f"\nZ-score (|z| > 3) sobre log1p: {int(atipicos_z.sum()):,} registros")
display(df.loc[atipicos_z, "numero_vehiculos"].value_counts().sort_index())

# Decisión explícita
DECISION_ATIPICOS = "conservar"  # opciones: "conservar" | "eliminar"
if DECISION_ATIPICOS == "eliminar":
    df = df.loc[~atipicos_z].reset_index(drop=True)
print(f"Decisión para numero_vehiculos: {DECISION_ATIPICOS}. Registros: {len(df):,}")

[numero_vehiculos] Q1=2.0, Q3=2.0, IQR=0.0
[numero_vehiculos] Rango: [2.0, 2.0] → atípicos: 28,253 (7.62%)

Z-score (|z| > 3) sobre log1p: 20,351 registros


numero_vehiculos
1.0     19062
4.0      1070
5.0       141
6.0        49
7.0        10
8.0        12
9.0         5
12.0        1
16.0        1
Name: count, dtype: int64

Decisión para numero_vehiculos: conservar. Registros: 370,998


## 8. Variables derivadas de vehículos y fecha

- **Vehículos:** cada tipo de vehículo se convierte en un indicador binario por categoría. Así no se crea una columna por cada uno de los cientos de textos distintos. Un valor `SIN_DATO` queda con todos los indicadores en 0.
- **Fecha:** año, mes, trimestre, día de la semana y fin de semana.

In [20]:
# Indicadores de tipo de vehículo implicado
vehiculos = df["vehiculos_texto"]
df["involucra_motocicleta"] = vehiculos.str.contains("MOTO", na=False).astype("int8")
df["involucra_liviano"] = vehiculos.str.contains("AUTOMOVIL|TAXI|CAMPERO|CAMIONETA", na=False).astype("int8")
df["involucra_pesado"] = vehiculos.str.contains("CAMION|BUS|MICROBUS|TRACTOCAMION", na=False).astype("int8")
df["involucra_no_motorizado"] = vehiculos.str.contains("BICI", na=False).astype("int8")

# Variables temporales
df["anio"] = df["fecha"].dt.year.astype("int16")
df["mes"] = df["fecha"].dt.month.astype("int8")
df["trimestre"] = df["fecha"].dt.quarter.astype("int8")
df["dia_semana"] = df["fecha"].dt.day_name()  # nominal
df["es_fin_de_semana"] = (df["fecha"].dt.dayofweek >= 5).astype("int8")

print(df[["involucra_motocicleta", "involucra_liviano", "involucra_pesado", "involucra_no_motorizado"]].mean().round(3))

involucra_motocicleta      0.184
involucra_liviano          0.316
involucra_pesado           0.087
involucra_no_motorizado    0.004
dtype: float64


## 9. Codificación

- **Ordinal** para `categoria_siniestro` (1 < 2 < 3 o más). Tiene orden natural, así que el código debe respetarlo.
- **One-Hot** con `drop_first=True` para las nominales (`tipo_accidente`, `medio_reporte`, `tipo_via_principal`, `dia_semana`). Se elimina una categoría de referencia para evitar la multicolinealidad perfecta. Con `drop_first`, la referencia es la primera categoría en orden alfabético.

In [21]:
# Ordinal: el tamaño del siniestro tiene orden natural
df["categoria_siniestro"] = np.select(
    [df["numero_vehiculos"] <= 1, df["numero_vehiculos"] == 2],
    ["1_VEHICULO", "2_VEHICULOS"],
    default="3_O_MAS",
)
codificador_ordinal = OrdinalEncoder(categories=[["1_VEHICULO", "2_VEHICULOS", "3_O_MAS"]])
df["categoria_siniestro_ord"] = (
    codificador_ordinal.fit_transform(df[["categoria_siniestro"]]).ravel().astype("int8")
)

display(
    df[["categoria_siniestro", "categoria_siniestro_ord"]]
    .drop_duplicates()
    .sort_values("categoria_siniestro_ord")
    .reset_index(drop=True)
)

,categoria_siniestro,categoria_siniestro_ord
0,1_VEHICULO,0
1,2_VEHICULOS,1
2,3_O_MAS,2


## 10. Transformación logarítmica

La transformación se aplica solo a variables muy sesgadas. Criterio: si |asimetría| > 1, se aplica `log1p` (segura con ceros). La asimetría se reporta antes y después para verificar el efecto.

In [22]:
UMBRAL_SESGO = 1.0
sesgo_antes = df["numero_vehiculos"].skew()

if abs(sesgo_antes) > UMBRAL_SESGO:
    df["numero_vehiculos_transf"] = np.log1p(df["numero_vehiculos"])
    print(f"Asimetría {sesgo_antes:.2f} > {UMBRAL_SESGO}: se aplica log1p")
else:
    df["numero_vehiculos_transf"] = df["numero_vehiculos"]
    print(f"Asimetría {sesgo_antes:.2f} <= {UMBRAL_SESGO}: no se transforma")

print(f"Asimetría de numero_vehiculos_transf: {df['numero_vehiculos_transf'].skew():.2f}")

Asimetría 1.52 > 1.0: se aplica log1p
Asimetría de numero_vehiculos_transf: -1.51


## 11. Escalado: criterio de selección

Se comparan tres escaladores sobre la variable transformada. Es un diagnóstico descriptivo: **no se ajusta ni se guarda en esta fase** (ver la regla de no fuga al inicio).

- **RobustScaler** depende del IQR. Aquí el IQR de la variable transformada es 0, así que solo centraría los datos y no reescalaría nada.
- **Min-Max** lleva el rango a [0, 1]. Con extremos conservados, la mayoría de los valores queda comprimida en un intervalo estrecho.
- **Z-score** (elegido) centra en la media y divide por la desviación estándar. Sobre la variable transformada, los extremos conservados tienen menos peso relativo en la desviación estándar.

**Aplicación en la Fase C:** `ColumnTransformer([("escalado", StandardScaler(), ["numero_vehiculos_transf"])])` dentro de un `Pipeline`, ajustado solo con el conjunto de entrenamiento.

In [23]:
valores = df[["numero_vehiculos_transf"]].to_numpy()
escaladores = {
    "Min-Max": MinMaxScaler(),
    "Z-score": StandardScaler(),
    "RobustScaler": RobustScaler(),
}

q1_log, q3_log = np.percentile(valores, [25, 75])
print(f"IQR de numero_vehiculos_transf: {q3_log - q1_log:.4f}\n")

for nombre, escalador in escaladores.items():
    escalado = escalador.fit_transform(valores).ravel()
    q1, mediana, q3 = np.percentile(escalado, [25, 50, 75])
    print(f"{nombre:>12}: mediana={mediana:+.3f} | Q1={q1:+.3f} | Q3={q3:+.3f} | rango=[{escalado.min():+.2f}, {escalado.max():+.2f}]")

ESCALADOR_ELEGIDO = "Z-score"
COLUMNAS_A_ESCALAR = ["numero_vehiculos_transf"]
print(f"\nEscalador elegido: {ESCALADOR_ELEGIDO} → se ajusta en Fase C sobre: {COLUMNAS_A_ESCALAR}")

IQR de numero_vehiculos_transf: 0.0000

     Min-Max: mediana=+0.189 | Q1=+0.189 | Q3=+0.189 | rango=[+0.00, +1.00]
     Z-score: mediana=+0.121 | Q1=+0.121 | Q3=+0.121 | rango=[-3.71, +16.53]
RobustScaler: mediana=+0.000 | Q1=+0.000 | Q3=+0.000 | rango=[-0.41, +1.73]

Escalador elegido: Z-score → se ajusta en Fase C sobre: ['numero_vehiculos_transf']


## 12. Selección final, validaciones y guardado

Se eliminan las columnas crudas y de alta cardinalidad (texto de dirección, texto de vehículos, gravedad cruda y `numero_vehiculos` ya transformada). El código de siniestro se conserva como identificador, no como predictor. Antes de guardar se verifica que no haya fuga de información, nulos ni duplicados.

In [24]:
columnas_nominales = ["tipo_accidente", "medio_reporte", "tipo_via_principal", "dia_semana"]
columnas_numericas = [
    "anio", "mes", "trimestre", "es_fin_de_semana", "es_interseccion",
    "numero_vehiculos_transf", "categoria_siniestro_ord",
    "involucra_motocicleta", "involucra_liviano", "involucra_pesado", "involucra_no_motorizado",
]

df_final = df[["Código", "fecha"] + columnas_nominales + columnas_numericas + [COLUMNA_OBJETIVO]].copy()
df_final = df_final.rename(columns={"Código": "id_siniestro"})
df_final["id_siniestro"] = df_final["id_siniestro"].astype("string")

# One-Hot Encoding con drop_first=True
df_final = pd.get_dummies(df_final, columns=columnas_nominales, drop_first=True, dtype="int8")

# Validaciones
columnas_prohibidas = {"Tipo confirmado", "Tipo confirmado.1", "tipo_confirmado", "gravedad_texto"}
assert not (columnas_prohibidas & set(df_final.columns)), "Hay columnas que filtran la variable objetivo"
assert df_final["id_siniestro"].is_unique, "Hay códigos de siniestro duplicados"
assert df_final["fecha"].notna().all(), "Hay fechas nulas"
assert set(df_final[COLUMNA_OBJETIVO].unique()) <= {0, 1}, "El objetivo no es binario"
assert df_final.drop(columns=["id_siniestro", "fecha"]).isna().sum().sum() == 0, "Quedan nulos"
print("Validaciones superadas.")

df_final.to_parquet(RUTA_SALIDA, index=False)

verificacion = pd.read_parquet(RUTA_SALIDA)
print(f"\nGuardado en: {RUTA_SALIDA.resolve()}")
print(f"Tamaño: {RUTA_SALIDA.stat().st_size / 1024**2:.2f} MB")
print(f"Dimensiones: {verificacion.shape[0]:,} filas × {verificacion.shape[1]} columnas")
print("\nColumnas finales:")
print(verificacion.columns.tolist())
print("\nDistribución de la variable objetivo:")
display(verificacion[COLUMNA_OBJETIVO].value_counts(normalize=True).round(4))

Validaciones superadas.

Guardado en: /home/santiago/Documentos/Universidad/Ciencia de Datos/Ciencias-de-datos/p2p3/entrega/datos_limpios.parquet
Tamaño: 2.27 MB
Dimensiones: 370,998 filas × 38 columnas

Columnas finales:
['id_siniestro', 'fecha', 'anio', 'mes', 'trimestre', 'es_fin_de_semana', 'es_interseccion', 'numero_vehiculos_transf', 'categoria_siniestro_ord', 'involucra_motocicleta', 'involucra_liviano', 'involucra_pesado', 'involucra_no_motorizado', 'con_victimas', 'tipo_accidente_CAIDA_OCUPANTE', 'tipo_accidente_CHOQUE', 'tipo_accidente_INCENDIO', 'tipo_accidente_OTRO', 'tipo_accidente_SIN_DATO', 'tipo_accidente_VOLCAMIENTO', 'medio_reporte_LINEA_123', 'medio_reporte_LINEA_127', 'medio_reporte_OTRO', 'medio_reporte_RADIO', 'medio_reporte_REDES_SOCIALES', 'medio_reporte_SIN_DATO', 'tipo_via_principal_CALLE', 'tipo_via_principal_CARRERA', 'tipo_via_principal_DIAGONAL', 'tipo_via_principal_REFERENCIA', 'tipo_via_principal_SIN_DATO', 'tipo_via_principal_TRANSVERSAL', 'dia_semana_M

con_victimas
0    0.8629
1    0.1371
Name: proportion, dtype: float64

## Resumen de decisiones por columna

| Columna original / derivada | Decisión | Justificación |
|---|---|---|
| `Código` | Identificador, no predictor. Se eliminan vacíos y duplicados (conserva el registro más completo). | Un siniestro debe aparecer una sola vez. |
| `Fecha` | Formato explícito `%d/%m/%Y`. Se derivan año, mes, trimestre, día y fin de semana. | El formato inferido anterior dejaba la mayoría de fechas en NaT. |
| `Tipo confirmado` / `.1` | Se construye `con_victimas` (0/1). Los textos sin gravedad se eliminan. Gravedad cruda fuera del modelo. | Es la variable objetivo. Usarla como predictor sería fuga de información. |
| `Tipo clase de accidente` / `.1` | Se toma la columna principal (coinciden 100 %). Estandarización a 6 categorías + `SIN_DATO`. One-Hot con `drop_first`. | Nominal. El nulo es informativo. |
| `Medio de reporte` | Unifica variantes de escritura en 6 categorías + `SIN_DATO`. One-Hot con `drop_first`. | Nominal con ~78 % de nulos y textos inconsistentes. |
| `Dirección reporte` | Se extraen tipo de vía y si es intersección. El texto crudo sale del modelo. | Cerca de 59 mil valores distintos: One-Hot directo no es viable. |
| `Tipo de vehículos implicados` | Indicadores binarios por categoría de vehículo. | Evita una columna por cada texto distinto. |
| `Numero vehículos` | Recuperación por regla de negocio, luego mediana. Atípicos conservados. Transformación `log1p` si |asimetría| > 1. | Discreta y sesgada. La mediana es robusta. Los extremos son siniestros reales. |
| `categoria_siniestro` | Ordinal (1 < 2 < 3 o más). | Tiene orden natural. |
| Escalado (`numero_vehiculos_transf`) | Z-score, ajustado dentro del Pipeline de Fase C. | Evita data leakage. |
| `dia_semana` | One-Hot con `drop_first`. | Nominal sin orden. |